# Step 4: Data preprocessing

From the parsed CWRU bearing data (step 1), we segment the time series of each bearing cycle, followed by computing the spectrograms.

The specs follow:
- https://www.sciencedirect.com/science/article/pii/S0888327021010499
- https://arxiv.org/abs/2407.14625

The resulting data is called the 'preprocessed' data.


In [ ]:
# Load the "autoreload" extension so that code can change
%load_ext autoreload
# Always reload modules so that as you change code in src, it gets loaded
%autoreload 2

import os
from copy import deepcopy
import json
import yaml
import re

import numpy as np
import pandas as pd
import scipy
from scipy.signal import ShortTimeFFT
import librosa

import matplotlib.pyplot as plt

from data_preprocessing import (
    create_sequences, compute_fft, compute_stft,
    plot_spectrogram, build_sequence_spectrogram, build_sequence_spectrum,
    convert_window_hop, get_window_fn, window_fn_names, preprocess_data_fftspectrum,
    preprocess_data_spectrogram)


## FFT (to compute discrete Fourier transform)

In [123]:
fs = 44100.  # sampling frequency

# simulated signal: a 440Hz sine wave followed by a 256Hz sine wave
N0 = int(7. * fs)
t0 = np.arange(N0) / fs
f0 = 440.
x0 = np.sin(2 * np.pi * f0 * t0)

N1 = int(7. * fs)
t1 = np.arange(N0, N0 + N1) / fs
f1 = 256.
x1 = np.sin(2 * np.pi * f1 * t1)

x = np.hstack([x0, x1])

In [ ]:
x.shape

In [ ]:
N0

In [126]:
fft_freq = np.linspace(0, fs * (N0 - 1) / N0, N0)

In [ ]:
fft_freq.shape

In [131]:
fft = scipy.fft.fft(np.vstack([x0, x1]).T, axis=0)

In [ ]:
fft.shape

In [ ]:
plt.plot(fft_freq, np.abs(fft[:,0]))
plt.xlim(0, 1000)

## Test short-time Fourier transform (STFT) and spectrogram

In [2]:
fs = 44100.  # sampling frequency

# simulated signal: a 440Hz sine wave followed by a 256Hz sine wave
N0 = int(10. * fs)
t0 = np.arange(N0) / fs
f0 = 440.
x0 = np.sin(2 * np.pi * f0 * t0)

N1 = int(10. * fs)
t1 = np.arange(N0, N0 + N1) / fs
f1 = 256.
x1 = np.sin(2 * np.pi * f1 * t1)

x = np.hstack([x0, x1])
x_twoch = np.vstack([x0, x1]).T

In [ ]:
x_twoch.shape

In [4]:
window_length_sec = 0.025  # frame (=window=sequence) size in seconds
hop = int(0.01 * fs)  #  = stride in the number of timesteps. Here we use 0.01s
mfft = n_fft = int(window_length_sec * fs)  # n_fft = number of data points in each frame (window) = frame size
window_fn = scipy.signal.windows.hamming(int(n_fft / 2))  # window function

In [5]:
# create scipy STFT object
SFT = scipy.signal.ShortTimeFFT(win=window_fn, fs=fs,
                                hop=hop, mfft=mfft)

In [6]:
stft = SFT.stft(x_twoch, axis=0)

In [ ]:
stft.shape

In [ ]:
stft[:,0:1,:].squeeze().shape

In [ ]:
x_twoch.shape

In [ ]:
SFT.stft(x0, axis=0).shape

In [ ]:
stft.shape

In [14]:
abs_stft = np.abs(stft)
# The Fourier frequencies are given by a rescaling as follows
# stft_freq = np.arange(0, int(1 + n_fft / 2)) * fs / n_fft
stft_freq = SFT.f
# Time steps
T0 = 0.
stft_time = T0 + np.arange(stft.shape[-1]) * SFT.delta_t

In [ ]:
plot_spectrogram(input_data=abs_stft[:,1,:],
                 stft_time=stft_time,
                 stft_freq=stft_freq,
                 input_is_spectrogram=True,
                 ylim=(0, 1000))

In [7]:
stft = SFT.stft(x)

In [ ]:
stft.shape

In [ ]:
# first dimension equals int(1 + n_fft / 2) for a real signal, to be rescaled to the Fourier frequencies
int(1 + n_fft / 2)

In [ ]:
# second dimension equals number of frames ~ number of strides modulo padding at the ends
len(x) / hop

In [ ]:
len(x) / fs

In [ ]:
np.hstack([t0, t1])

In [ ]:
(np.arange(stft.shape[1]) * hop) / fs

In [37]:
abs_stft = np.abs(stft)
# The Fourier frequencies are given by a rescaling as follows
# stft_freq = np.arange(0, int(1 + n_fft / 2)) * fs / n_fft
stft_freq = SFT.f
# Time steps
T0 = 0.
stft_time = T0 + np.arange(stft.shape[1]) * SFT.delta_t

In [ ]:
def convert_window_hop(window_length_sec: float,
                       hop_sec: float,
                       fs: float):
    mfft = n_fft = int(window_length_sec * fs)  # n_fft = number of data points in each frame (window) = frame size
    hop = int(hop_sec * fs)  #  = stride in the number of timesteps. Here we use 0.01s
    return mfft, hop

In [158]:
def compute_stft(x: np.ndarray,
                 fs: float,
                 mfft: int, hop: int,
                 window_fn=None,
                 return_spectrogram: bool = True,
                 start_time: float = 0.,
                 ):
    '''Computes short-time Fourier transform (STFT) of a given possible
    multivariate signal, using scipy.

    Args:
        x (np.ndarray): a univariate/multivariate signal.
            If multivariate, x.shape == (signal_length, num_channels)
        fs (float): sampling frequency in Hz
        mfft (int): = n_fft = frame size
            = number of data points in each frame (window)
        hop (int): hop = stride in the number of timesteps
        window_fn: a scipy window function.
            Defaults to None, in which case we use
            scipy.signal.windows.hann(mfft)
        return_spectrogram (bool, optional):
            return spectrogram, otherwise return stft.
            Defaults to True.
        start_time (float, optional): start time of the signal.
            Defaults to 0.

    Returns:
        stft_freq (np.ndarray): Fourier frequencies of the STFT
        stft_time (np.ndarray): time steps of the STFT
        spectrogram (np.ndarray) if return_spectrogram,
        else stft (np.ndarray)
        SFT: scipy.signal.ShortTimeFFT object, for possible later use
            e.g. computing cepstrum
    '''
    x_ = x
    if x.ndim == 1:
        x_ = np.expand_dims(x, axis=-1)
    if not window_fn:
        window_fn = scipy.signal.windows.hann(mfft)  # window function
    # create scipy STFT object
    SFT = scipy.signal.ShortTimeFFT(win=window_fn, fs=fs,
                                    hop=hop, mfft=mfft)
    # compute stft of the given signal
    stft = SFT.stft(x_, axis=0)
    # The Fourier frequencies are given by a rescaling as follows
    # stft_freq = np.arange(0, int(1 + n_fft / 2)) * fs / n_fft
    stft_freq = SFT.f
    # time steps
    stft_time = start_time + np.arange(stft.shape[-1]) * SFT.delta_t
    if return_spectrogram:
        spectrogram = np.square(np.abs(stft))
        return stft_freq, stft_time, spectrogram, SFT
    else:
        return stft_freq, stft_time, stft, SFT

In [ ]:
plot_spectrogram(input_data=stft,
                 stft_time=stft_time,
                 stft_freq=stft_freq,
                 input_is_spectrogram=False,
                 ylim=(0, 600))

Compare against librosa's STFT

In [ ]:
librosa.stft(x0, n_fft=n_fft, hop_length=hop,
             window=scipy.signal.windows.hann(n_fft)).shape

In [39]:
stft2 = librosa.stft(x, n_fft=n_fft, hop_length=hop)

In [ ]:
stft2.shape

In [ ]:
librosa.display.specshow(np.abs(stft2), sr=fs, y_axis='hz', x_axis='time')
plt.ylim(0,600)
plt.colorbar()

## Cepstrum

In [ ]:
x

In [57]:
cepstrum = np.square(SFT.istft(
    np.log10(np.square(abs_stft)),
    k0=0, k1=x.shape[0]))

In [ ]:
cepstrum.shape

In [ ]:
plt.plot(np.hstack([t0, t1]), cepstrum)
plt.ylim(90, 140)
plt.xlim(10, 11)

## Sequence construction

In [30]:
fs = 44100.  # sampling frequency

# simulated signal: a 440Hz sine wave followed by a 256Hz sine wave
N2 = int(10. * fs)
t2 = np.arange(N2) / fs
f20 = 440.
x20 = np.sin(2 * np.pi * f20 * t2)

f21 = 256.
x21 = np.sin(2 * np.pi * f21 * t2)

x2 = np.vstack([x20, x21]).T

In [ ]:
x2.shape

In [32]:
sequence_length = 100000 # length of sequence in number of data points
sequence_stride = int(0.1 * sequence_length)


In [35]:
sequences, t_starts = create_sequences(x2,
                             sequence_length=sequence_length,
                             stride=sequence_stride)

In [ ]:
sequences.shape

### FFT for each sequence

In [ ]:
%%time
seq_fft = []
fft_freq = None
for n in range(sequences.shape[0]):
    x_ = sequences[n, ...]
    fft_freq_, fft_ = compute_fft(x_, fs=fs, return_spectrum=True,
                                  signal_is_real=False)
    seq_fft.append(fft_)
fft_freq = fft_freq_

In [ ]:
seq_fft[0].shape

In [ ]:
plt.plot(fft_freq, seq_fft[0])
plt.xlim(0, 1000)

### Spectrogram for each sequence

In [20]:
window_length_sec = 0.025  # frame (=window=sequence) size in seconds
hop = int(0.01 * fs)  #  = stride in the number of timesteps. Here we use 0.01s
mfft = n_fft = int(window_length_sec * fs)  # n_fft = number of data points in each frame (window) = frame size
window_fn = scipy.signal.windows.hann(n_fft)  # window function


In [32]:
seq_spectrogram = []
stft_freq = None
stft_time = None
for n in range(sequences.shape[0]):
    x_ = sequences[n, ...]
    stft_freq_, stft_time_, spectrogram, SFT = \
        compute_stft(x_,
                     fs=fs, mfft=mfft, hop=hop,
                     window_fn=window_fn,
                     return_spectrogram=True)
    seq_spectrogram.append(spectrogram)
stft_freq = stft_freq_
stft_time = stft_time_

In [ ]:
len(seq_spectrogram)

In [ ]:
plot_spectrogram(seq_spectrogram[30],
                 stft_freq=stft_freq,
                 stft_time=stft_time,
                 plot_channel=1,
                 ylim=(0,1000))

## Bearing example

In [9]:
TS_COL = 'timestamp'
fs = 12000
df0 = pd.read_csv('./data_split/central/train/97.csv').sort_values(TS_COL)


In [ ]:
df0

In [ ]:
1 / df0[TS_COL].diff()

In [12]:
data_columns = ["DE_time", "FE_time"]
CYCLE_ID = 'cycle_id'

In [19]:
specs = yaml.load('''
fft_spectrum_specs:
  fs: 12000
  signal_is_real: True  # use rfft and returns half of the spectrum by symmetry
  sequence_length: 4095 # length of sequence in number of data points
  sequence_stride: 122 # stride of sequence in number of data points
spectrogram_specs:
  fs: 12000
  sequence_length: 11500 # length of sequence in number of data points
  sequence_stride: 345 # stride of sequence in number of data points
  hop: 54  #  = stride in the number of timesteps. Here we use 0.01s
  mfft: 452  # n_fft = number of data points in each frame = frame size
  # window function
  window_fn_name: "hann"
  window_fn_args:
    M: 104
  channel_first: True  # otherwise channel is on axis=1 from scipy. For PyTorch use
                  ''',
                  yaml.SafeLoader)

In [ ]:
preprocess_data_fftspectrum()

In [15]:
x0 = df0[data_columns].to_numpy()
t0 = df0[TS_COL].to_numpy()

In [24]:
sequences, t_starts\
 = create_sequences(x=x0, sequence_length= 4095,
                     stride= 122, padding = True,
                     pad_mode= 'median',
                     t=t0)

In [16]:
fft_freq, seq_spectrum = build_sequence_spectrum(x=x0, **specs['fft_spectrum_specs'])

In [67]:
seq_fft = []
fft_freq = None
for n in range(sequences.shape[0]):
    x_ = sequences[n, ...]
    fft_freq_, fft_ = compute_fft(x_, fs=fs,
                                  signal_is_real=True,
                                  return_spectrum=True)
    seq_fft.append(fft_)
fft_freq = fft_freq_

In [ ]:
fft_freq.shape

In [ ]:
seq_fft[0].shape

In [ ]:
np.stack(seq_fft).shape

In [ ]:
plt.plot(fft_freq[:seq_fft[0].shape[0]], seq_fft[0][:,0])
plt.yscale('log')

In [ ]:
yaml.load('''

preprocessing_specs:
  # sequence segmentation specs
  sequence_length: 11500 # length of sequence in number of data points
  sequence_stride: 345 # stride of sequence in number of data points
  compute_fft_spectrum: True
  fft_spectrum_specs: null
  compute_spectrogram: True
  spectrogram_specs:
    hop: 54  #  = stride in the number of timesteps. Here we use 0.01s
    mfft: 452  # n_fft = number of data points in each frame = frame size
    # window function
    window_fn_name: "hann"
    window_fn_args:
      M: 104
    channel_first: True  # otherwise channel is on axis=1 from scipy. For PyTorch use

          ''', yaml.SafeLoader)

In [36]:
fs = cycle_['sampling_freq_khz'] * 1000

In [43]:
sequence_length = 11500 # length of sequence in number of data points
sequence_stride = int(0.03 * sequence_length)
hop = 54  #  = stride in the number of timesteps. Here we use 0.01s
mfft = n_fft = 452  # n_fft = number of data points in each frame (window) = frame size
window_fn_name = 'hann'
window_fn_args = {'M': 104}

In [44]:
stft_freq, stft_time, seq_spectrogram = build_sequence_spectrogram(
    x=x0,
    sequence_length=sequence_length, sequence_stride=sequence_stride,
    fs=fs, mfft=mfft, hop=hop,
    window_fn_name=window_fn_name, window_fn_args=window_fn_args
)

In [ ]:
seq_spectrogram[0].shape

In [66]:
np.savez('test.npz', stft_freq=stft_freq, stft_time=stft_time, seq_spectrogram=seq_spectrogram)

In [67]:
npzfiles = np.load('test.npz')

In [ ]:
npzfiles.values()

In [71]:
stft_freq = npzfiles['stft_freq']
stft_time = npzfiles['stft_time']
seq_spectrogram = npzfiles['seq_spectrogram']

In [ ]:
plot_spectrogram(seq_spectrogram[0].swapaxes(0,1),
                 stft_freq=stft_freq,
                 stft_time=stft_time,
                 plot_channel=0,
                #  ylim=(0,1000)
                 )

In [39]:
fft_spec = np.load('./data_fftspectrum/client_0/train/98.npz')

In [ ]:
fft_spec['seq_spectrum'].shape

In [44]:
spectro_spec = np.load('./data_spectrogram/client_0/train/98.npz')

In [ ]:
for i in range(10):
    plot_spectrogram(spectro_spec['seq_spectrogram'][i],
                    stft_freq=spectro_spec['stft_freq'],
                    stft_time=spectro_spec['stft_time'],
                    plot_channel=1,
                    ylim=(0,1000))

In [ ]:
plt.pcolormesh(stft_time, stft_freq, input_data_[plot_channel, :])

In [68]:
split_dir = './data_split'

In [76]:
source_data_dirs = [
    os.path.join(split_dir, dir_)
    for dir_ in os.listdir(split_dir)
    if os.path.isdir(os.path.join(split_dir, dir_))]

In [ ]:
os.path.split('./data_split/client_5/train')

In [ ]:
os.path.normpath('./data_split/client_5/train').split(os.path.sep)

In [81]:
subsets = ['train', 'val', 'test']
filepath_fields = ['train_data_path', 'val_data_path', 'test_data_path']

In [ ]:
[(x, y) for x in source_data_dirs for y in subsets]

In [1]:
import numpy as np

In [21]:
a = np.load('/home/cedricyu/projects/ml_experiment_cwu_bearing/ml-experiments/cwu_bearings/data_spectrogram/client_0/test/98.npz')

In [ ]:
list(a.keys())

In [ ]:
a['seq_spectrogram'].shape

In [ ]:
a['seq_spectrum'].shape